## 1. REPARTITION()
This is the "start from scratch" method.
* **What it does:** It completely reshuffles your data to create the exact number of partitions you ask for.
* **Direction:** Can increase or decrease the number of partitions.
* **The Catch:** It triggers a Full Shuffle. This means data is sent across the network between different servers (executors).
* **Best for:**
    * Increasing parallelism (e.g., going from 10 partitions to 100)
    * Fixing "Data Skew" (when some partitions are huge and others are empty)
        *

## 2. COALESCE()
This is the "optimized shrink" method.
* **What it does:** It reduces the number of partitions by merging existing ones together.
* **Direction:** Only used to decrease the number of partitions.
* **The Advantage:** It avoids a full shuffle. It tries to keep data on the same nodes as much as possible.
* **Best for:**
    * Cleaning up after a big filter (e.g., you filtered out 90% of your data and now have too many empty partitions)
    * Saving files (e.g., merging 1,000 small files into 10 before writing to S3 or HDFS)

In [1]:
# importing important libs related to spark
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, StructField, StructType, DoubleType

In [2]:
# creating spark object for SS Builder() => Class and builder => object | builder showing warning.
spark = SparkSession.Builder().master("local[*]").appName("repartitionVsCoalesce").getOrCreate()


In [3]:
# creating Schema for DF
df_schema = StructType(
    [
    StructField("longitude", DoubleType(), nullable=False),
    StructField("latitude", DoubleType(), nullable=False),
    StructField("housing_median_age", DoubleType(), nullable=False),
    StructField("total_rooms", DoubleType(), nullable=False),
    StructField("total_bedrooms", DoubleType(), nullable=False),
    StructField("population", DoubleType(), nullable=False),
    StructField("households", DoubleType(), nullable=False),
    StructField("median_income", DoubleType(), nullable=False),
    StructField("median_house_value", DoubleType(), nullable=False),
    StructField("ocean_proximity", StringType(), nullable=False)
    ]
)

# creating DF
df = spark.read.format("csv") \
    .option("inferSchema", "false") \
    .option("header", "true") \
    .schema(df_schema) \
    .load("airbnb.csv")

In [4]:
# checking the number of partition in DF
print(f"number of partition before repartition = {df.rdd.getNumPartitions()}")

number of partition before repartition = 1


In [5]:
# increasing the number of partition from 1 to 10
df = df.repartition(10)

In [6]:
# checking the number of partition in DF
print(f"number of partition after repartition = {df.rdd.getNumPartitions()}")

number of partition after repartition = 10


In [7]:
# reducing the number for partition using coalesce
df = df.coalesce(2)

In [8]:
# checking the number for partition after coalesce
print(f"number of partition after coalesce = {df.rdd.getNumPartitions()}")

number of partition after coalesce = 2


In [12]:
spark.stop()